In [1]:
#!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 73.1 MB/s eta 0:00:00


In [3]:
from gensim.models import FastText

In [4]:
corpus=[
    "The quick brown for jumps over the lazy dog",
    "I love learning natural language processing",
    "FastText is a powerful word embedding model",
    "Dogs are better than cats for some helps",
    "I enjoy solving problems with machine learning"
]

tokenized = [sentence.lower().split() for sentence in corpus]

cbow_model = FastText(sentences=tokenized, vector_size=100, window=5, min_count=1, sg=0, epochs=1)

In [6]:
word="learning"
print({word}, cbow_model.wv[word][:10])

{'learning'} [ 1.0893568e-03 -5.3324849e-05 -9.4723265e-04  3.9520164e-04
 -1.9701642e-03 -4.8359664e-04 -5.7009672e-04 -6.7526847e-04
  1.3388217e-03  3.7089287e-04]


In [7]:
cbow_model.wv.most_similar("learning", topn=5)

[('embedding', 0.249241441488266),
 ('a', 0.19803036749362946),
 ('some', 0.18332931399345398),
 ('processing', 0.15252277255058289),
 ('jumps', 0.1313648223876953)]

In [8]:
from gensim.models import Word2Vec

sg_ns_model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, sg=1, negative=10, epochs=1)

In [9]:
print(word, sg_ns_model.wv[word][:10])

learning [-0.00053623  0.00023643  0.00510335  0.00900927 -0.00930295 -0.00711681
  0.00645887  0.00897299 -0.00501543 -0.00376337]


In [12]:
print(sg_ns_model.wv.most_similar("dog", topn=3))

[('enjoy', 0.2855507433414459), ('i', 0.18890604376792908), ('jumps', 0.14239901304244995)]


In [13]:
## document rating

In [14]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np


In [15]:
data = [
    ("I loved the movie, it was fantastic!", "positive", 4.5),
    ("The movie was okay, not the best.", "neutral", 3.0),
    ("Terrible movie. Wouldn't recommend.", "negative", 1.2),
    ("Really enjoyed the characters and plot.", "positive", 4.7),
    ("It was boring and predictable.", "negative", 2.0),
    ("An average film, not too bad.", "neutral", 3.2),
]

In [16]:
sentences=[text.lower().split() for text, _, _ in data]
scores = [score for _,_, score in data]

In [17]:
sentences

[['i', 'loved', 'the', 'movie,', 'it', 'was', 'fantastic!'],
 ['the', 'movie', 'was', 'okay,', 'not', 'the', 'best.'],
 ['terrible', 'movie.', "wouldn't", 'recommend.'],
 ['really', 'enjoyed', 'the', 'characters', 'and', 'plot.'],
 ['it', 'was', 'boring', 'and', 'predictable.'],
 ['an', 'average', 'film,', 'not', 'too', 'bad.']]

In [18]:
scores

[4.5, 3.0, 1.2, 4.7, 2.0, 3.2]

In [20]:
# train the model
w2v_model = Word2Vec(sentences=sentences, vector_size=50, window=5, min_count=1, workers=2, epochs=100)

In [21]:
def sentence_vector(tokens, model):
  vectors=[model.wv[word] for word in tokens if word in model.wv]
  if vectors:
    return np.mean(vectors, axis=0)
  else:
    return np.zeros(model.vector_size)

In [22]:
x=[sentence_vector(tokens, w2v_model) for tokens in sentences]
y=scores

In [23]:
reg=Ridge()
reg.fit(x,y)

Ridge()

In [24]:
y_pred = reg.predict(x)
mse=mean_squared_error(y, y_pred)
print("Mean Squared Error:", mse)

Mean Squared Error: 1.5555136288283364


In [27]:
new_review ="I loved the first heart of films"
new_tokens = new_review.lower().split()
new_vec = sentence_vector(new_tokens, w2v_model)
predicted_score = reg.predict([new_vec])
print("Predicted Score:", predicted_score[0])

Predicted Score: 3.101395763543579
